# EDA — Análisis Exploratorio del Dataset Gold

Este notebook analiza el dataset sintético generado para DocShield.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.dataset.generator import generate_fraud_dataset
from src.dataset.labeler import apply_heuristics

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Generar dataset
print('Generando dataset sintético...')
df = generate_fraud_dataset(n_legit=4000, n_fraud=600)
df = apply_heuristics(df)

print(f'Total muestras: {len(df)}')
print(f'Fraude: {df["is_fraud"].sum()} ({df["is_fraud"].mean()*100:.1f}%)')
print('\nDistribución por tipo:')
print(df['fraud_type'].value_counts())

In [ ]:
# Guardar dataset Gold
import os
os.makedirs('../data/gold', exist_ok=True)
df.to_parquet('../data/gold/dataset.parquet', index=False)
print('Dataset Gold guardado en data/gold/dataset.parquet')

In [ ]:
# Análisis de distribuciones por clase
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
features_to_plot = ['blur_score', 'ela_score', 'ocr_confidence', 
                     'moire_score', 'dct_score', 'ip_risk_score',
                     'brightness', 'contrast', 'noise_ratio']

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx//3, idx%3]
    for label, color in [(0, 'green'), (1, 'red')]:
        data = df[df['is_fraud'] == label][feature]
        ax.hist(data, alpha=0.6, bins=30, label=f'{"Fraud" if label==1 else "Legit"}', color=color)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frecuencia')
    ax.legend()
    ax.set_title(f'Distribución: {feature}')

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(12, 10))
corr_features = ['blur_score', 'edge_density', 'brightness', 'contrast', 'noise_ratio',
                'symmetry_score', 'color_variance', 'ela_score', 'moire_score',
                'dct_score', 'reflection_score', 'ocr_confidence', 'ip_risk_score',
                'is_fraud']

corr = df[corr_features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de Correlación de Features')
plt.tight_layout()
plt.show()

In [ ]:
# Análisis por tipo de fraude
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fraud_types = df[df['is_fraud']==1]['fraud_type'].unique()

for idx, ftype in enumerate(fraud_types):
    ax = axes[idx//2, idx%2]
    subset = df[df['fraud_type'] == ftype]
    ax.scatter(subset['blur_score'], subset['ela_score'], alpha=0.6, label=ftype)
    ax.set_xlabel('blur_score')
    ax.set_ylabel('ela_score')
    ax.set_title(f'Tipo: {ftype}')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas descriptivas
print('=== Estadísticas por clase ===')
for cls in [0, 1]:
    label = 'FRAUDE' if cls == 1 else 'LEGIT'
    print(f'\n--- {label} ---')
    subset = df[df['is_fraud'] == cls]
    print(subset[['blur_score', 'ela_score', 'ocr_confidence', 'moire_score']].describe().round(2))